# GroundingDINO + SAM2 segmentation benchmark (API 3D)

Notebook này đánh giá đúng worker `worker_sam2_dino.py` đang dùng bởi `api-3d.ipynb`. GroundingDINO tự tìm bounding box từ nhãn lớp; SAM2 và rembg/matting dùng nguyên logic production. Ground truth lấy từ COCO 2017 validation.

**Cách chạy:** chạy `api-3d.ipynb` đến khi hiện `SAM2_DINO WORKER READY`, sau đó chạy toàn bộ notebook này trong cùng Kaggle session. Không cần chạy SD3.5 hoặc TRELLIS.

Chỉ số xuất ra: Detection Recall/TPR tại IoU 0.5, Precision, mean Box IoU, mask IoU, Dice, Boundary F1, tỷ lệ RGBA hợp lệ, tỷ lệ lỗi, tỷ lệ detector fallback và tỷ lệ rembg refinement.

In [ ]:
# 0. MATCH THE API-3D RUNTIME (PYTHON 3.10 + TORCH 2.1 + CUDA 12.1)
# This cell must run before the benchmark configuration cell.
# It prevents the benchmark from falling back to Kaggle system Python 3.12.
import os, shutil, subprocess, sys
from pathlib import Path

VENV_ROOT = Path('/opt/venv310')
BENCHMARK_PYTHON = VENV_ROOT / 'bin' / 'python'

def run_checked(command, label, env=None):
    print(f'[{label}]', ' '.join(map(str, command)))
    result = subprocess.run(command, env=env, capture_output=True, text=True)
    if result.returncode != 0:
        print((result.stdout or '')[-2000:])
        print((result.stderr or '')[-6000:])
        raise RuntimeError(f'{label} failed with exit code {result.returncode}')
    return result

if not BENCHMARK_PYTHON.is_file() or subprocess.run([str(BENCHMARK_PYTHON), '-m', 'pip', '--version'], capture_output=True).returncode != 0:
    python310 = shutil.which('python3.10')
    if python310 is None:
        run_checked(['apt-get', 'update', '-qq'], 'apt update')
        run_checked(['apt-get', 'install', '-y', '-qq', 'software-properties-common'], 'install apt helpers')
        run_checked(['add-apt-repository', '-y', 'ppa:deadsnakes/ppa'], 'add Python repository')
        run_checked(['apt-get', 'update', '-qq'], 'apt update after repository')
        run_checked(['apt-get', 'install', '-y', '-qq', 'python3.10', 'python3.10-dev', 'python3.10-venv', 'build-essential', 'git', 'git-lfs'], 'install API runtime')
        python310 = shutil.which('python3.10')
    else:
        run_checked(['apt-get', 'update', '-qq'], 'apt update')
        run_checked(['apt-get', 'install', '-y', '-qq', 'python3.10-venv', 'python3.10-dev', 'build-essential', 'git', 'git-lfs'], 'install Python 3.10 venv support')
    if not python310:
        raise FileNotFoundError('python3.10 is unavailable after installation')
    if VENV_ROOT.exists():
        shutil.rmtree(VENV_ROOT)
    run_checked([python310, '-m', 'venv', str(VENV_ROOT)], 'create /opt/venv310')

if not BENCHMARK_PYTHON.is_file():
    raise FileNotFoundError(f'API runtime was not created: {BENCHMARK_PYTHON}')

PYTHON = str(BENCHMARK_PYTHON)
PIP = [PYTHON, '-m', 'pip']

runtime_probe = subprocess.run([PYTHON, '-c', "import torch, torchvision, diffusers, transformers, peft, accelerate; print(torch.__version__); print(torch.cuda.is_available()); print(torchvision.__version__); print(diffusers.__version__); print(transformers.__version__); print(peft.__version__); print(accelerate.__version__)"], capture_output=True, text=True)
probe_lines = [line.strip() for line in runtime_probe.stdout.splitlines() if line.strip()]
web_probe = subprocess.run([PYTHON, '-c', "import fastapi, pydantic, multipart, uvicorn, requests; print('web runtime ok')"], capture_output=True, text=True)
runtime_ok = (runtime_probe.returncode == 0 and web_probe.returncode == 0 and len(probe_lines) >= 7 and probe_lines[0].startswith('2.1.0') and probe_lines[1] == 'True' and probe_lines[2].startswith('0.16.0') and probe_lines[3] == '0.32.2' and probe_lines[4] == '4.46.3' and probe_lines[5] == '0.10.0' and probe_lines[6] == '0.34.2')

if not runtime_ok:
    if runtime_probe.returncode != 0:
        print('Existing /opt/venv310 is incomplete; installing the exact api-3d pins.')
        print((runtime_probe.stderr or runtime_probe.stdout)[-3000:])
        if web_probe.returncode != 0:
            print('Web runtime is incomplete; installing FastAPI/Uvicorn dependencies.')
            print((web_probe.stderr or web_probe.stdout)[-3000:])
    run_checked([*PIP, 'install', '-q', '--upgrade', 'pip<25'], 'pin pip')
    run_checked([*PIP, 'install', '-q', 'torch==2.1.0', 'torchvision==0.16.0', 'xformers==0.0.22.post7', '--index-url', 'https://download.pytorch.org/whl/cu121'], 'install API PyTorch stack')
    api_packages = [
        'numpy==1.26.4', 'pillow==10.4.0', 'diffusers==0.32.2',
        'transformers==4.46.3', 'peft==0.10.0', 'accelerate==0.34.2',
        'huggingface_hub==0.34.6', 'hydra-core>=1.3.2', 'iopath>=0.1.10',
        'timm==0.9.16', 'supervision>=0.22.0', 'addict', 'yapf',
        'pycocotools', 'opencv-python-headless', 'scipy', 'rembg[cpu]==2.0.60',
        'onnxruntime==1.20.1', 'setuptools==69.5.1', 'wheel', 'ninja',
        'fastapi', 'pydantic', 'python-multipart', 'uvicorn', 'requests',
    ]
    run_checked([*PIP, 'install', '-q', *api_packages], 'install API dependency pins')

version_check = run_checked([PYTHON, '-c', "import numpy, torch, torchvision, diffusers, transformers; print('numpy', numpy.__version__); print('torch', torch.__version__); print('torchvision', torchvision.__version__); print('diffusers', diffusers.__version__); print('transformers', transformers.__version__); print('cuda', torch.cuda.is_available())"], 'final API runtime check')
print(version_check.stdout)
if '2.1.0' not in version_check.stdout or 'True' not in version_check.stdout:
    raise RuntimeError('The benchmark runtime does not match api-3d. Stop before compiling GroundingDINO.')
print('API-compatible runtime ready:', PYTHON)


In [ ]:
# 1. CONFIGURATION AND PREFLIGHT
import csv, json, math, os, random, shutil, statistics, subprocess, sys, time, zipfile
from collections import Counter, defaultdict
from pathlib import Path

import requests

SEED = 42
WORKER_URL = 'http://127.0.0.1:8003'
START_STANDALONE_WORKER = True  # Benchmark can prepare and start SAM2/DINO itself
OUTPUT_ROOT = Path('/kaggle/working/segmentation_benchmark_api3d')
COCO_ROOT = OUTPUT_ROOT / 'coco'
IMAGE_ROOT = COCO_ROOT / 'val2017'
PRED_ROOT = OUTPUT_ROOT / 'predictions'
REPORT_ROOT = OUTPUT_ROOT / 'report'
TARGET_COUNTS = {'chair': 34, 'couch': 33, 'dining table': 33}
# COCO labels are kept for reporting; these aliases are only used as
# GroundingDINO text queries because the API vocabulary uses sofa/table.
DINO_LABEL_ALIASES = {'chair': 'chair', 'couch': 'sofa', 'dining table': 'table'}
CATEGORY_IDS = {'chair': 62, 'couch': 63, 'dining table': 67}
BOX_IOU_THRESHOLD = 0.50
ALPHA_THRESHOLD = 32
BOUNDARY_TOLERANCE = 2
MIN_AREA_RATIO = 0.015
REQUEST_TIMEOUT = 180
READY_TIMEOUT = 600
WORKER_LOG = Path('/kaggle/working/segmentation_benchmark_sam2_dino_worker.log')
WORKER_PROC = None
WORKER_PYTHON = '/opt/venv310/bin/python'
if not Path(WORKER_PYTHON).is_file():
    raise RuntimeError('API runtime missing. Run cell 0 first; benchmark refuses Python 3.12 fallback.')
EXTENSIONS_ROOT = Path('/kaggle/working/vendor')
GROUNDING_ROOT = EXTENSIONS_ROOT / 'GroundingDINO'
SAM2_ROOT = EXTENSIONS_ROOT / 'sam2'
COMPAT_ROOT = EXTENSIONS_ROOT / 'compat'

for path in (IMAGE_ROOT, PRED_ROOT, REPORT_ROOT):
    path.mkdir(parents=True, exist_ok=True)

def get_health():
    response = requests.get(f'{WORKER_URL}/health', timeout=5)
    response.raise_for_status()
    return response.json()

def wait_for_ready(timeout_seconds=READY_TIMEOUT):
    last_health = None
    for second in range(timeout_seconds):
        try:
            last_health = get_health()
            if last_health.get('error'):
                raise RuntimeError(
                    'SAM2_DINO worker failed while loading models:\n'
                    + json.dumps(last_health, indent=2, ensure_ascii=False)
                )
            if last_health.get('ready'):
                return last_health
        except requests.RequestException:
            pass
        if (second + 1) % 10 == 0:
            print(f'Waiting for SAM2_DINO worker... {second + 1}/{timeout_seconds}s')
        time.sleep(1)
    raise RuntimeError(
        'SAM2_DINO worker did not become ready. Last health: '
        + json.dumps(last_health, indent=2, ensure_ascii=False)
    )

def clone_source(url, destination, required_relative_path):
    destination = Path(destination)
    required_file = destination / required_relative_path
    if destination.exists() and required_file.is_file():
        return destination
    if destination.exists():
        shutil.rmtree(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    print(f'Cloning {destination.name} source...')
    clone = subprocess.run(
        ['git', 'clone', '--depth', '1', url, str(destination)],
        capture_output=True, text=True,
    )
    if clone.returncode != 0:
        raise RuntimeError(f'Clone failed for {url}:\n' + (clone.stderr or clone.stdout)[-5000:])
    return destination

def segmentation_env():
    # Kaggle's notebook kernel exports matplotlib's inline backend.
    # GroundingDINO is a headless worker, so force the API notebook's
    # non-GUI backend for every subprocess and never inherit module://... .
    os.environ['MPLBACKEND'] = 'Agg'
    # Both projects are pure-Python at import time for this worker.
    # Use their checked-out source directly; editable installation invokes
    # GroundingDINO's legacy setup.py and fails on current Kaggle images.
    env = os.environ.copy()
    env['MPLBACKEND'] = 'Agg'
    env['CUDA_HOME'] = '/usr/local/cuda'
    env['PATH'] = '/usr/local/cuda/bin:' + env.get('PATH', '')
    env['PYTHONPATH'] = os.pathsep.join([
        str(COMPAT_ROOT), str(GROUNDING_ROOT), str(SAM2_ROOT),
        '/kaggle/working', env.get('PYTHONPATH', ''),
    ])
    return env

def write_transformers_compat():
    COMPAT_ROOT.mkdir(parents=True, exist_ok=True)
    sitecustomize = COMPAT_ROOT / 'sitecustomize.py'
    sitecustomize.write_text(r'''
import torch
try:
    from transformers import BertModel
    if not hasattr(BertModel, '_convert_head_mask_to_5d'):
        def _convert_head_mask_to_5d(self, head_mask, num_hidden_layers):
            if head_mask.dim() == 1:
                head_mask = head_mask[None, None, :, None, None]
                head_mask = head_mask.expand(num_hidden_layers, -1, -1, -1, -1)
            elif head_mask.dim() == 2:
                head_mask = head_mask[:, None, :, None, None]
            if head_mask.dim() != 5:
                raise ValueError(f'head_mask.dim()={head_mask.dim()}, expected 5')
            return head_mask
        BertModel._convert_head_mask_to_5d = _convert_head_mask_to_5d
    if not hasattr(BertModel, 'get_head_mask'):
        def _get_head_mask(self, head_mask, num_hidden_layers, is_attention_chunked=False):
            if head_mask is not None:
                head_mask = self._convert_head_mask_to_5d(head_mask, num_hidden_layers)
                if is_attention_chunked:
                    head_mask = head_mask.unsqueeze(-1)
            else:
                head_mask = [None] * num_hidden_layers
            return head_mask
        BertModel.get_head_mask = _get_head_mask
    if not hasattr(BertModel, 'get_extended_attention_mask'):
        def _get_extended_attention_mask(self, attention_mask, input_shape, device=None, dtype=None):
            if attention_mask.dim() == 3:
                extended_attention_mask = attention_mask[:, None, :, :]
            elif attention_mask.dim() == 2:
                extended_attention_mask = attention_mask[:, None, None, :]
            else:
                raise ValueError(f'Wrong shape for attention_mask: {tuple(attention_mask.shape)}')
            if device is not None:
                extended_attention_mask = extended_attention_mask.to(device)
            dtype = dtype or self.dtype
            extended_attention_mask = extended_attention_mask.to(dtype=dtype)
            return (1.0 - extended_attention_mask) * torch.finfo(dtype).min
        BertModel.get_extended_attention_mask = _get_extended_attention_mask
except Exception as exc:
    print('Transformers compatibility patch skipped:', exc)
''', encoding='utf-8')

write_transformers_compat()

SEGMENTATION_RUNTIME_PACKAGES = [
    'hydra-core>=1.3.2',
    'iopath>=0.1.10',
    'timm==0.9.16',
    'supervision>=0.22.0',
    'addict',
    'yapf',
    'pycocotools',
    'opencv-python-headless',
    'scipy',
    'pillow',
    'rembg[cpu]==2.0.60',
    'onnxruntime==1.20.1',
]

SEGMENTATION_IMPORT_PROBE = (
    'from groundingdino.util.inference import load_image, load_model, predict\n'
    'from sam2.build_sam import build_sam2\n'
    'from sam2.sam2_image_predictor import SAM2ImagePredictor'
)

def run_segmentation_import_probe(env):
    return subprocess.run(
        [WORKER_PYTHON, '-c', SEGMENTATION_IMPORT_PROBE],
        env=env, capture_output=True, text=True,
    )

def ensure_segmentation_packages():
    clone_source(
        'https://github.com/IDEA-Research/GroundingDINO.git',
        GROUNDING_ROOT,
        'groundingdino/__init__.py',
    )
    clone_source(
        'https://github.com/facebookresearch/sam2.git',
        SAM2_ROOT,
        'sam2/__init__.py',
    )
    import_env = segmentation_env()
    verify = run_segmentation_import_probe(import_env)
    if verify.returncode != 0:
        print('Installing missing segmentation runtime dependencies...')
        install = subprocess.run(
            [WORKER_PYTHON, '-m', 'pip', 'install', '-q',
             *SEGMENTATION_RUNTIME_PACKAGES],
            env=import_env, capture_output=True, text=True,
        )
        if install.returncode != 0:
            raise RuntimeError(
                'Cannot install segmentation runtime dependencies.\n'
                + (install.stderr or install.stdout)[-10000:]
            )
        verify = run_segmentation_import_probe(import_env)
    if verify.returncode != 0:
        raise RuntimeError(
            'Cannot import GroundingDINO/SAM2 from source.\n'
            'The legacy editable-install step is intentionally disabled.\n'
            + (verify.stderr or verify.stdout)[-10000:]
        )
    print('GroundingDINO and SAM2 loaded directly from source.')

ensure_segmentation_packages()

# GroundingDINO imports as Python source, but its multi-scale deformable
# attention also needs the compiled groundingdino._C extension.
# Build it explicitly before starting the worker; otherwise the first
# /segment request fails with NameError: name '_C' is not defined.
def ensure_groundingdino_extension():
    build_env = segmentation_env()
    build_env['TORCH_CUDA_ARCH_LIST'] = '7.5'  # Tesla T4
    build_env['MAX_JOBS'] = '2'
    build_env['FORCE_CUDA'] = '1'
    setup_py = GROUNDING_ROOT / 'setup.py'
    if not setup_py.is_file():
        raise FileNotFoundError(f'GroundingDINO setup.py not found: {setup_py}')

    probe = (
        'import groundingdino._C as extension; '
        'print("GroundingDINO extension ready:", extension.__file__)'
    )
    ready = subprocess.run(
        [WORKER_PYTHON, '-c', probe],
        env=build_env, capture_output=True, text=True,
    )
    if ready.returncode == 0:
        print(ready.stdout.strip())
        return

    print('Building GroundingDINO CUDA extension...')
    build_commands = [
        [WORKER_PYTHON, '-m', 'pip', 'install', '-q', '--no-deps',
         '--no-build-isolation', '-e', str(GROUNDING_ROOT)],
        [WORKER_PYTHON, 'setup.py', 'build_ext', '--inplace'],
    ]
    build_logs = []
    for command in build_commands:
        build = subprocess.run(
            command, cwd=str(GROUNDING_ROOT), env=build_env,
            capture_output=True, text=True,
        )
        build_logs.append((build.stdout or '')[-5000:] + (build.stderr or '')[-10000:])
        if build.returncode == 0:
            verify_build = subprocess.run(
                [WORKER_PYTHON, '-c', probe], env=build_env,
                capture_output=True, text=True,
            )
            if verify_build.returncode == 0:
                print(verify_build.stdout.strip())
                return
        print('GroundingDINO build attempt failed; trying the fallback build command...')
    raise RuntimeError(
        'GroundingDINO CUDA extension build failed.\n'
        + '\n'.join(build_logs)
    )

ensure_groundingdino_extension()

# Always synchronize the worker so a previous Kaggle run cannot keep a
# stale version of the API worker after a backend fix.
FORCE_SYNC_WORKER = True
worker_script = Path('/kaggle/working/worker_sam2_dino.py')
if FORCE_SYNC_WORKER or not worker_script.is_file():
    REPOSITORY_URL = 'https://raw.githubusercontent.com/Tiens0710/DATN-3d/0cddf46/worker_sam2_dino.py'
    print('Synchronizing refined worker commit 0cddf46 from GitHub...')
    response = requests.get(REPOSITORY_URL, timeout=120)
    response.raise_for_status()
    worker_script.write_bytes(response.content)
    worker_text = worker_script.read_text(encoding='utf-8')
    if (
        worker_script.stat().st_size < 1000
        or 'DINO_RUNTIME_DEVICE' not in worker_text
        or 'SAM_BOX_PADDING_RATIO' not in worker_text
        or 'SAM_POINT_REFINEMENT' not in worker_text
    ):
        raise RuntimeError('Downloaded worker is incomplete or does not contain commit 0cddf46 improvements')
    print('Backend source synchronized:', worker_script)

def download_if_missing(path, url, minimum_size):
    path = Path(path)
    if path.is_file() and path.stat().st_size >= minimum_size:
        return
    from urllib.request import urlretrieve
    path.parent.mkdir(parents=True, exist_ok=True)
    print(f'Downloading {path.name}...')
    urlretrieve(url, path)
    if not path.is_file() or path.stat().st_size < minimum_size:
        raise RuntimeError(f'Invalid or incomplete download: {path}')

def ensure_segmentation_checkpoints():
    dino_dir = Path('/kaggle/working/groundingdino_ckpt')
    sam2_dir = Path('/kaggle/working/sam2_ckpt')
    dino_config = dino_dir / 'GroundingDINO_SwinT_OGC.py'
    dino_weights = dino_dir / 'groundingdino_swint_ogc.pth'
    sam2_weights = sam2_dir / 'sam2_hiera_small.pt'
    if not dino_config.is_file():
        source_config = GROUNDING_ROOT / 'groundingdino' / 'config' / 'GroundingDINO_SwinT_OGC.py'
        if not source_config.is_file():
            raise FileNotFoundError(source_config)
        dino_dir.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source_config, dino_config)
    download_if_missing(
        dino_weights,
        'https://github.com/IDEA-Research/GroundingDINO/releases/download/v0.1.0-alpha/groundingdino_swint_ogc.pth',
        100_000_000,
    )
    download_if_missing(
        sam2_weights,
        'https://dl.fbaipublicfiles.com/segment_anything_2/072824/sam2_hiera_small.pt',
        100_000_000,
    )
    print('Segmentation checkpoints ready.')

ensure_segmentation_checkpoints()

# A standalone worker can now start even when api-3d.ipynb was not run.
REQUIRED_WORKER_FILES = [
    Path('/kaggle/working/groundingdino_ckpt/GroundingDINO_SwinT_OGC.py'),
    Path('/kaggle/working/groundingdino_ckpt/groundingdino_swint_ogc.pth'),
    Path('/kaggle/working/sam2_ckpt/sam2_hiera_small.pt'),
]

def stop_old_worker():
    subprocess.run(['pkill', '-f', 'worker_sam2_dino.py.*--port 8003'], check=False)
    time.sleep(2)

def start_clean_worker():
    global WORKER_PROC
    worker_python = WORKER_PYTHON
    WORKER_LOG.parent.mkdir(parents=True, exist_ok=True)
    log_handle = WORKER_LOG.open('w', buffering=1, encoding='utf-8')
    worker_env = segmentation_env()
    worker_env['MPLBACKEND'] = 'Agg'
    worker_env.setdefault('ENABLE_MATTING', '1')
    worker_env['MASK_MIN_COMPONENT_RATIO'] = '0.00002'
    worker_env['SAM_BOX_PADDING_RATIO'] = '0.05'
    worker_env['SAM_BOX_PADDING_MAX'] = '96'
    worker_env['SAM_ALPHA_BLUR_SIGMA'] = '0.35'
    worker_env['SAM_POINT_REFINEMENT'] = '1'
    worker_env['PYTHONPATH'] = os.pathsep.join([
        str(GROUNDING_ROOT), str(SAM2_ROOT),
        worker_env.get('PYTHONPATH', ''),
    ])
    WORKER_PROC = subprocess.Popen(
        [worker_python, '-u', str(worker_script), '--host', '127.0.0.1', '--port', '8003'],
        cwd='/kaggle/working', stdout=log_handle, stderr=subprocess.STDOUT, env=worker_env,
    )
    print(f'SAM2_DINO worker started, pid={WORKER_PROC.pid}. Waiting for model load...')
    for second in range(READY_TIMEOUT):
        if WORKER_PROC.poll() is not None:
            log_handle.flush()
            raise RuntimeError('SAM2_DINO worker stopped:\n' + WORKER_LOG.read_text(encoding='utf-8', errors='replace')[-12000:])
        try:
            candidate = get_health()
            if candidate.get('error'):
                log_handle.flush()
                raise RuntimeError(
                    'SAM2_DINO worker failed while loading models:\n'
                    + json.dumps(candidate, indent=2, ensure_ascii=False)
                    + '\nWorker log:\n'
                    + WORKER_LOG.read_text(encoding='utf-8', errors='replace')[-12000:]
                )
            if candidate.get('ready'):
                return candidate
        except requests.RequestException:
            pass
        if (second + 1) % 10 == 0:
            print(f'Waiting for SAM2_DINO... {second + 1}/{READY_TIMEOUT}s')
        time.sleep(1)
    raise RuntimeError('SAM2_DINO worker timeout:\n' + WORKER_LOG.read_text(encoding='utf-8', errors='replace')[-12000:])

try:
    health_data = get_health()
except requests.RequestException:
    health_data = None

if FORCE_SYNC_WORKER:
    print('Worker source was synchronized; restarting worker to load the new code.')
    stop_old_worker()
    health_data = start_clean_worker()
elif health_data and health_data.get('ready'):
    print('Using existing SAM2_DINO worker.')
elif health_data and not health_data.get('error'):
    print('Existing worker is still loading; waiting for readiness...')
    try:
        health_data = wait_for_ready(180)
    except RuntimeError:
        print('Existing worker did not become ready; restarting it cleanly.')
        stop_old_worker()
        health_data = start_clean_worker()
else:
    if health_data:
        print('Existing SAM2_DINO worker is unhealthy; restarting it cleanly.')
    else:
        print('No SAM2_DINO worker found; starting it automatically.')
    stop_old_worker()
    health_data = start_clean_worker()

print(json.dumps(health_data, indent=2, ensure_ascii=False))
if not health_data.get('ready'):
    raise RuntimeError('SAM2_DINO worker chưa sẵn sàng. Hãy chạy api-3d.ipynb đến SAM2_DINO WORKER READY.')
print('Benchmark output:', OUTPUT_ROOT)

In [ ]:
# 2. INSTALL LIGHTWEIGHT EVALUATION DEPENDENCIES
required = {'pycocotools': 'pycocotools', 'matplotlib': 'matplotlib'}
for module_name, package_name in required.items():
    try:
        __import__(module_name)
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', package_name], check=True)

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image, ImageDraw, ImageFilter
from IPython.display import display
from pycocotools import mask as mask_utils
print('Evaluation dependencies ready.')

In [ ]:
# 3. DOWNLOAD COCO ANNOTATIONS AND BUILD A FIXED 100-IMAGE TEST SET
ANNOTATION_ZIP = COCO_ROOT / 'annotations_trainval2017.zip'
ANNOTATION_FILE = COCO_ROOT / 'annotations' / 'instances_val2017.json'
ANNOTATION_URL = 'https://images.cocodataset.org/annotations/annotations_trainval2017.zip'

COCO_VERIFY_SSL = True
COCO_SSL_WARNING_SHOWN = False

def coco_request(url, *, stream=False, timeout=120):
    # Kaggle may reject COCO's certificate. Retry without verification once
    # and reuse that decision so every image does not print the warning.
    global COCO_VERIFY_SSL, COCO_SSL_WARNING_SHOWN
    if not COCO_VERIFY_SSL:
        response = requests.get(url, stream=stream, timeout=timeout, verify=False)
        response.raise_for_status()
        return response
    try:
        response = requests.get(url, stream=stream, timeout=timeout, verify=True)
        response.raise_for_status()
        return response
    except requests.exceptions.SSLError:
        import urllib3
        urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
        COCO_VERIFY_SSL = False
        if not COCO_SSL_WARNING_SHOWN:
            print('COCO SSL certificate failed; continuing with verification disabled for this session...')
            COCO_SSL_WARNING_SHOWN = True
        response = requests.get(url, stream=stream, timeout=timeout, verify=False)
        response.raise_for_status()
        return response

if not ANNOTATION_FILE.is_file():
    if not ANNOTATION_ZIP.is_file():
        print('Downloading official COCO 2017 annotations...')
        with coco_request(ANNOTATION_URL, stream=True, timeout=120) as response:
            with ANNOTATION_ZIP.open('wb') as file:
                for chunk in response.iter_content(1024 * 1024):
                    if chunk:
                        file.write(chunk)
    if not zipfile.is_zipfile(ANNOTATION_ZIP):
        ANNOTATION_ZIP.unlink(missing_ok=True)
        raise RuntimeError('COCO annotation download is not a valid ZIP file.')
    with zipfile.ZipFile(ANNOTATION_ZIP) as archive:
        archive.extractall(COCO_ROOT)

with ANNOTATION_FILE.open(encoding='utf-8') as file:
    coco = json.load(file)
images_by_id = {item['id']: item for item in coco['images']}
annotations_by_category = defaultdict(lambda: defaultdict(list))
for annotation in coco['annotations']:
    if annotation['category_id'] in CATEGORY_IDS.values() and not annotation.get('iscrowd', 0):
        annotations_by_category[annotation['category_id']][annotation['image_id']].append(annotation)

rng = random.Random(SEED)
samples, used_image_ids = [], set()
for label, target_count in TARGET_COUNTS.items():
    category_id = CATEGORY_IDS[label]
    candidates = []
    for image_id, annotations in annotations_by_category[category_id].items():
        if len(annotations) != 1:
            continue
        image_info = images_by_id[image_id]
        annotation = annotations[0]
        area_ratio = float(annotation['area']) / max(1, image_info['width'] * image_info['height'])
        if area_ratio >= MIN_AREA_RATIO:
            candidates.append((image_info, annotation, area_ratio))
    rng.shuffle(candidates)
    chosen = 0
    for image_info, annotation, area_ratio in candidates:
        if image_info['id'] in used_image_ids:
            continue
        sample_id = f"{label.replace(' ', '_')}_{image_info['id']}"
        samples.append({'sample_id': sample_id, 'label': label, 'image': image_info, 'annotation': annotation, 'area_ratio': area_ratio})
        used_image_ids.add(image_info['id'])
        chosen += 1
        if chosen >= target_count:
            break
    if chosen < target_count:
        raise RuntimeError(f'COCO chỉ chọn được {chosen}/{target_count} mẫu cho {label}')

for index, sample in enumerate(samples, start=1):
    file_name = sample['image']['file_name']
    image_path = IMAGE_ROOT / file_name
    if not image_path.is_file():
        url = f'https://images.cocodataset.org/val2017/{file_name}'
        response = coco_request(url, timeout=60)
        try:
            image_path.write_bytes(response.content)
        finally:
            response.close()
    sample['image_path'] = str(image_path)
    if index % 10 == 0:
        print(f'Downloaded {index}/{len(samples)} images')

protocol = {
    'dataset': 'COCO 2017 validation', 'seed': SEED, 'sample_count': len(samples),
    'target_counts': TARGET_COUNTS, 'classes': CATEGORY_IDS,
    'box_iou_threshold': BOX_IOU_THRESHOLD, 'alpha_threshold': ALPHA_THRESHOLD,
    'boundary_tolerance_px': BOUNDARY_TOLERANCE, 'min_area_ratio': MIN_AREA_RATIO,
    'worker_url': WORKER_URL, 'worker_health': health_data,
}
(REPORT_ROOT / 'protocol.json').write_text(json.dumps(protocol, indent=2, ensure_ascii=False), encoding='utf-8')
print('Test set:', Counter(sample['label'] for sample in samples))

In [ ]:
# 4. METRIC FUNCTIONS
def coco_mask(annotation, height, width):
    segmentation = annotation['segmentation']
    if isinstance(segmentation, list):
        rles = mask_utils.frPyObjects(segmentation, height, width)
        rle = mask_utils.merge(rles)
    else:
        rle = segmentation
        if isinstance(rle.get('counts'), list):
            rle = mask_utils.frPyObjects(rle, height, width)
    decoded = mask_utils.decode(rle)
    if decoded.ndim == 3:
        decoded = np.any(decoded, axis=2)
    return decoded.astype(bool)

def xywh_to_xyxy(box):
    x, y, width, height = map(float, box)
    return [x, y, x + width, y + height]

def box_iou(first, second):
    ax1, ay1, ax2, ay2 = first; bx1, by1, bx2, by2 = second
    intersection = max(0.0, min(ax2, bx2) - max(ax1, bx1)) * max(0.0, min(ay2, by2) - max(ay1, by1))
    area_a = max(0.0, ax2 - ax1) * max(0.0, ay2 - ay1)
    area_b = max(0.0, bx2 - bx1) * max(0.0, by2 - by1)
    return intersection / max(1e-9, area_a + area_b - intersection)

def mask_scores(prediction, target):
    prediction, target = prediction.astype(bool), target.astype(bool)
    intersection = np.logical_and(prediction, target).sum()
    union = np.logical_or(prediction, target).sum()
    iou = intersection / max(1, union)
    dice = 2 * intersection / max(1, prediction.sum() + target.sum())
    def erode(mask):
        return np.asarray(Image.fromarray(mask.astype(np.uint8) * 255).filter(ImageFilter.MinFilter(3))) > 0
    def dilate(mask):
        size = 2 * BOUNDARY_TOLERANCE + 1
        return np.asarray(Image.fromarray(mask.astype(np.uint8) * 255).filter(ImageFilter.MaxFilter(size))) > 0
    pred_boundary = np.logical_xor(prediction, erode(prediction))
    gt_boundary = np.logical_xor(target, erode(target))
    pred_near_gt = np.logical_and(pred_boundary, dilate(gt_boundary)).sum()
    gt_near_pred = np.logical_and(gt_boundary, dilate(pred_boundary)).sum()
    precision = pred_near_gt / max(1, pred_boundary.sum())
    recall = gt_near_pred / max(1, gt_boundary.sum())
    boundary_f1 = 2 * precision * recall / max(1e-9, precision + recall)
    return float(iou), float(dice), float(boundary_f1)

def valid_rgba(path):
    image = Image.open(path)
    if image.mode != 'RGBA':
        return False
    alpha = np.asarray(image.getchannel('A'))
    return bool(alpha.max() > 0 and alpha.min() < 255)

In [ ]:
# 5. SMOKE TEST THE SAME /segment WORKER USED BY API 3D
smoke_sample = samples[0]
smoke_dir = PRED_ROOT / '_smoke_test'
smoke_dir.mkdir(parents=True, exist_ok=True)
smoke_payload = {
    'input_image_path': smoke_sample['image_path'],
    'objects': [{'id': 'smoke_test', 'label': DINO_LABEL_ALIASES[smoke_sample['label']]}],
    'crops_dir': str(smoke_dir), 'source_mode': 'uploaded', 'auto_detect': False,
}
smoke_response = requests.post(f'{WORKER_URL}/segment', json=smoke_payload, timeout=REQUEST_TIMEOUT)
if not smoke_response.ok:
    try:
        smoke_detail = smoke_response.json().get('detail', smoke_response.json())
    except ValueError:
        smoke_detail = smoke_response.text
    raise RuntimeError(f'SAM2/DINO smoke test failed: HTTP {smoke_response.status_code}: {smoke_detail}')
smoke_result = smoke_response.json().get('results', [])
if not smoke_result or not Path(smoke_result[0]['crop_path']).is_file():
    raise RuntimeError(f'SAM2/DINO smoke test returned no valid crop: {smoke_response.text}')
print('SAM2/DINO smoke test passed:', smoke_result[0])

# 6. RUN THE SAME /segment WORKER USED BY API 3D
rows = []
for index, sample in enumerate(samples, start=1):
    sample_dir = PRED_ROOT / sample['sample_id']
    sample_dir.mkdir(parents=True, exist_ok=True)
    image_info, annotation = sample['image'], sample['annotation']
    gt_box = xywh_to_xyxy(annotation['bbox'])
    gt_mask = coco_mask(annotation, image_info['height'], image_info['width'])
    payload = {
        'input_image_path': sample['image_path'],
        'objects': [{'id': sample['sample_id'], 'label': DINO_LABEL_ALIASES[sample['label']]}],
        'crops_dir': str(sample_dir), 'source_mode': 'uploaded', 'auto_detect': False,
    }
    started = time.perf_counter()
    row = {
        'sample_id': sample['sample_id'], 'image_id': image_info['id'], 'label': sample['label'],
        'image_path': sample['image_path'], 'gt_box': json.dumps(gt_box), 'status': 'failed',
        'confidence': 0.0, 'box_iou': 0.0, 'box_tp_50': 0, 'mask_iou': 0.0,
        'dice': 0.0, 'boundary_f1': 0.0, 'valid_rgba': 0, 'detector_fallback': 0,
        'rembg_refined': 0, 'segmentation_method': 'failed', 'error': '',
    }
    try:
        response = requests.post(f'{WORKER_URL}/segment', json=payload, timeout=REQUEST_TIMEOUT)
        if not response.ok:
            try:
                body = response.json()
                detail = body.get('detail', body) if isinstance(body, dict) else body
            except ValueError:
                detail = response.text
            raise RuntimeError(
                f'worker /segment HTTP {response.status_code}: {detail}'
            )
        body = response.json()
        results = body.get('results') if isinstance(body, dict) else None
        if not results:
            raise RuntimeError(f'worker /segment returned no results: {body}')
        result = results[0]
        prediction_path = Path(result['crop_path'])
        if not prediction_path.is_file():
            raise FileNotFoundError(f'Worker crop not found: {prediction_path}')
        pred_rgba = Image.open(prediction_path).convert('RGBA')
        pred_alpha = np.asarray(pred_rgba.getchannel('A'))
        if pred_alpha.shape != gt_mask.shape:
            pred_alpha = np.asarray(Image.fromarray(pred_alpha).resize((image_info['width'], image_info['height']), Image.Resampling.NEAREST))
        pred_mask = pred_alpha >= ALPHA_THRESHOLD
        miou, dice, boundary_f1 = mask_scores(pred_mask, gt_mask)
        detected_box = list(map(float, result['box']))
        biou = box_iou(detected_box, gt_box)
        method = str(result.get('segmentation_method', 'sam2'))
        row.update({
            'status': 'ok', 'pred_box': json.dumps(detected_box),
            'confidence': float(result.get('confidence', 0.0)), 'box_iou': biou,
            'box_tp_50': int(biou >= BOX_IOU_THRESHOLD), 'mask_iou': miou, 'dice': dice,
            'boundary_f1': boundary_f1, 'valid_rgba': int(valid_rgba(prediction_path)),
            'detector_fallback': int(bool(result.get('detector_fallback', False))),
            'rembg_refined': int('isnet' in method.lower() or 'rembg' in method.lower()),
            'segmentation_method': method, 'prediction_path': str(prediction_path),
        })
    except Exception as error:
        row['error'] = str(error)[:500]
    row['latency_seconds'] = time.perf_counter() - started
    rows.append(row)
    print(f"[{index:03d}/{len(samples)}] {sample['label']:<12} status={row['status']:<6} boxIoU={row['box_iou']:.3f} maskIoU={row['mask_iou']:.3f}")

with (REPORT_ROOT / 'per_sample_metrics.csv').open('w', newline='', encoding='utf-8-sig') as file:
    writer = csv.DictWriter(file, fieldnames=list(rows[0].keys()))
    writer.writeheader(); writer.writerows(rows)
(REPORT_ROOT / 'per_sample_metrics.json').write_text(json.dumps(rows, indent=2, ensure_ascii=False), encoding='utf-8')
print('Raw results saved.')

In [ ]:
# 6. AGGREGATE, PLOTS AND THESIS-READY REPORT
def mean_of(frame, key):
    return float(sum(float(row[key]) for row in frame) / max(1, len(frame)))

def summarize(frame):
    total = len(frame); detected = [row for row in frame if row['status'] == 'ok']; matched = [row for row in frame if row['box_tp_50'] == 1]
    predictions = len(detected); true_positives = len(matched)
    return {
        'samples': total, 'predictions': predictions, 'true_positives_iou50': true_positives,
        'detection_recall_tpr_iou50': true_positives / max(1, total),
        'detection_precision_iou50': true_positives / max(1, predictions),
        'mean_box_iou_all': mean_of(frame, 'box_iou'),
        'mean_confidence_detected': mean_of(detected, 'confidence') if detected else 0.0,
        'mean_mask_iou_end_to_end': mean_of(frame, 'mask_iou'),
        'mean_mask_iou_matched_detections': mean_of(matched, 'mask_iou') if matched else 0.0,
        'mean_dice_end_to_end': mean_of(frame, 'dice'),
        'mean_boundary_f1_end_to_end': mean_of(frame, 'boundary_f1'),
        'valid_rgba_rate': mean_of(frame, 'valid_rgba'),
        'pipeline_failure_rate': sum(row['status'] != 'ok' for row in frame) / max(1, total),
        'detector_fallback_rate': mean_of(frame, 'detector_fallback'),
        'rembg_refinement_rate': mean_of(frame, 'rembg_refined'),
        'mean_latency_seconds': mean_of(frame, 'latency_seconds'),
    }

overall = summarize(rows)
per_class = {label: summarize([row for row in rows if row['label'] == label]) for label in sorted(TARGET_COUNTS)}
report = {'protocol': protocol, 'overall': overall, 'per_class': per_class}
(REPORT_ROOT / 'segmentation_metrics.json').write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding='utf-8')
summary_rows = [{'scope': 'overall', **overall}] + [{'scope': label, **metrics} for label, metrics in per_class.items()]
with (REPORT_ROOT / 'segmentation_metrics.csv').open('w', newline='', encoding='utf-8-sig') as file:
    writer = csv.DictWriter(file, fieldnames=list(summary_rows[0].keys()))
    writer.writeheader(); writer.writerows(summary_rows)

print(json.dumps(summary_rows, indent=2, ensure_ascii=False))
fig, axes = plt.subplots(1, 2, figsize=(16, 6), constrained_layout=False)
labels = list(per_class); x = np.arange(len(labels)); width = 0.24
recall_bars = axes[0].bar(x - width, [per_class[label]['detection_recall_tpr_iou50'] for label in labels], width, label='Detection TPR', color='#2563eb')
mask_bars = axes[0].bar(x, [per_class[label]['mean_mask_iou_end_to_end'] for label in labels], width, label='Mask IoU', color='#059669')
dice_bars = axes[0].bar(x + width, [per_class[label]['mean_dice_end_to_end'] for label in labels], width, label='Dice', color='#d97706')
axes[0].bar_label(recall_bars, fmt='%.2f', padding=3, fontsize=8)
axes[0].bar_label(mask_bars, fmt='%.2f', padding=3, fontsize=8)
axes[0].bar_label(dice_bars, fmt='%.2f', padding=3, fontsize=8)
axes[0].set_xticks(x); axes[0].set_xticklabels(labels, rotation=15, ha='right')
axes[0].set_ylim(0, 1.12); axes[0].set_title('Detection and segmentation by class', pad=10); axes[0].set_ylabel('Score')
axes[0].legend(loc='upper center', bbox_to_anchor=(0.5, -0.20), ncol=3, fontsize=9, frameon=True)
axes[0].grid(axis='y', alpha=.25); axes[0].set_axisbelow(True)
hist_bins = np.linspace(0.0, 1.0, 11)
axes[1].hist([row['box_iou'] for row in rows], bins=hist_bins, histtype='step', linewidth=2.2, label='Box IoU', color='#2563eb')
axes[1].hist([row['mask_iou'] for row in rows], bins=hist_bins, histtype='step', linewidth=2.2, label='Mask IoU', color='#059669')
axes[1].set_xlim(0, 1); axes[1].set_title('Metric distribution over 100 images', pad=10); axes[1].set_xlabel('Score'); axes[1].set_ylabel('Number of images')
axes[1].legend(loc='upper center', bbox_to_anchor=(0.5, -0.20), ncol=2, fontsize=9, frameon=True)
axes[1].grid(axis='y', alpha=.25); axes[1].set_axisbelow(True)
fig.subplots_adjust(top=.90, bottom=.34, left=.07, right=.98, wspace=.28); fig.savefig(REPORT_ROOT / 'segmentation_benchmark_chart.png', dpi=220, bbox_inches='tight'); plt.show()

print('IMPORTANT: rembg_refinement_rate is not a fallback rate. In the current API, rembg/isnet is an optional boundary refiner.')
print(json.dumps(overall, indent=2, ensure_ascii=False))

pct = lambda value: f'{100 * value:.2f}%'.replace('.', ',')
thesis_text = f'''4.4.2. Đánh giá chất lượng phân lập và bóc tách nền (Segmentation Quality)

Thí nghiệm được thực hiện trên {overall['samples']} ảnh thuộc tập COCO 2017 validation, gồm ghế, sofa và bàn ăn. Pipeline đánh giá sử dụng đúng GroundingDINO, SAM 2 và bộ tinh chỉnh biên rembg/isnet đang được triển khai trong API 3D.

GroundingDINO đạt Recall/TPR tại ngưỡng IoU 0,5 là {pct(overall['detection_recall_tpr_iou50'])}, Precision là {pct(overall['detection_precision_iou50'])}, với Box IoU trung bình {overall['mean_box_iou_all']:.4f}. Confidence trung bình trên các trường hợp phát hiện được là {overall['mean_confidence_detected']:.4f}; giá trị confidence này không được xem là độ chính xác.

Trên toàn bộ pipeline, mặt nạ đầu ra đạt mIoU {overall['mean_mask_iou_end_to_end']:.4f}, Dice {overall['mean_dice_end_to_end']:.4f} và Boundary F1 {overall['mean_boundary_f1_end_to_end']:.4f}. Nếu chỉ xét các vật thể được GroundingDINO định vị đúng tại IoU >= 0,5, mIoU của mặt nạ đạt {overall['mean_mask_iou_matched_detections']:.4f}.

Tỷ lệ ảnh RGBA hợp lệ là {pct(overall['valid_rgba_rate'])}; tỷ lệ pipeline thất bại là {pct(overall['pipeline_failure_rate'])}. Bộ rembg/isnet được chấp nhận để tinh chỉnh biên trong {pct(overall['rembg_refinement_rate'])} số mẫu. Đây là tỷ lệ tinh chỉnh biên, không phải tỷ lệ fallback khi GroundingDINO thất bại.'''
(REPORT_ROOT / 'thesis_section_4_4_2.txt').write_text(thesis_text, encoding='utf-8')
print('\n' + thesis_text)

In [ ]:
# 7. VISUAL AUDIT: BEST, MEDIAN AND WORST MASK IOU
ordered = sorted(rows, key=lambda row: row['mask_iou']); median_value = statistics.median(row['mask_iou'] for row in rows)
audit = ordered[:3] + sorted(rows, key=lambda row: abs(row['mask_iou'] - median_value))[:3] + ordered[-3:]
audit = list({row['sample_id']: row for row in audit}.values())
for row in audit:
    source = Image.open(row['image_path']).convert('RGB')
    draw = ImageDraw.Draw(source)
    gt_box = json.loads(row['gt_box']); draw.rectangle(gt_box, outline='lime', width=4)
    if row['status'] == 'ok':
        draw.rectangle(json.loads(row['pred_box']), outline='red', width=3)
    print(f"{row['sample_id']}: boxIoU={row['box_iou']:.3f}, maskIoU={row['mask_iou']:.3f}, Dice={row['dice']:.3f}; green=GT, red=prediction")
    display(source.resize((min(640, source.width), int(source.height * min(640, source.width) / source.width))))
    if row['status'] == 'ok' and Path(row['prediction_path']).is_file():
        display(Image.open(row['prediction_path']).convert('RGBA'))

archive = shutil.make_archive('/kaggle/working/segmentation_benchmark_api3d_report', 'zip', root_dir=REPORT_ROOT)
print('Report ZIP:', archive)
try:
    from IPython.display import FileLink
    display(FileLink(archive))
except Exception:
    pass

## Cách viết mục 4.4.2

Chỉ sử dụng các giá trị thật trong `report/segmentation_metrics.json`. TPR là `detection_recall_tpr_iou50`; chất lượng SAM2 nên báo cáo `mean_mask_iou_end_to_end`, `mean_dice_end_to_end` và `mean_boundary_f1_end_to_end`. Không gọi `rembg_refinement_rate` là fallback vì API hiện dùng rembg/isnet để tinh chỉnh biên khi kết quả tương thích với SAM2.